# Pavimentos-Brasil — Detecção de Objetos com YOLOv8

**Dataset:** [Pavimentos-Brasil](https://www.kaggle.com/datasets/mateusserafim/pavimentosbrasil)
**Objetivo:** Localizar e delimitar defeitos de pavimento com bounding boxes para cálculo de área afetada (ICM/DNIT)
**Modelo:** YOLOv8n (Ultralytics)
**Ambiente:** Google Colab — GPU T4

---

**Fluxo do projeto:**

```
Imagens brutas  -->  Anotação (Roboflow)  -->  Dataset YOLO  -->  Treino YOLOv8  -->  Avaliação  -->  Exportação
```

> **Pré-requisito:** Tenha seu `kaggle.json` e um projeto anotado exportado do Roboflow no formato YOLOv8. Se ainda não tiver anotações, a Seção 3 mostra como gerar pseudo-labels automaticamente como ponto de partida.

---
## Setup

In [ ]:

DATA_DIR         = '/content/pavimentos'
YOLO_DATA_DIR    = '/content/yolo_dataset'
MODEL_DIR        = '/content/drive/MyDrive/pavimentos_yolo'
ROBOFLOW_API_KEY = ''
ROBOFLOW_WS      = ''
ROBOFLOW_PROJECT = ''
ROBOFLOW_VERSION = 1
KAGGLE_DATASET   = 'mateusserafim/pavimentosbrasil'

CLASS_NAMES = ['buraco', 'remendo', 'trinca', 'sinalizacao_vertical',
               'sinalizacao_horizontal', 'drenagem', 'vegetacao']
NUM_CLASSES = len(CLASS_NAMES)

IMG_SIZE    = 640
BATCH_SIZE  = 16
NUM_EPOCHS  = 50
CONF_THRESH = 0.25
IOU_THRESH  = 0.45
SEED        = 42

YOLO_MODEL  = 'yolov8n.pt'

print('Configuracoes definidas.')

In [ ]:

!pip install -q ultralytics roboflow supervision opencv-python-headless

import ultralytics
ultralytics.checks()
print('Dependencias instaladas.')

In [ ]:
%matplotlib inline

import os, shutil, random, yaml, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import cv2
from pathlib import Path
from PIL import Image as PILImage # Renomeado para evitar conflito
from tqdm.notebook import tqdm
from glob import glob

import torch
from ultralytics import YOLO
import supervision as sv

random.seed(SEED)
np.random.seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')


In [ ]:

from google.colab import files, drive

drive.mount('/content/drive')
os.makedirs(MODEL_DIR, exist_ok=True)

print('Faca upload do seu kaggle.json:')
uploaded = files.upload()

kaggle_dir = Path('/root/.kaggle')
kaggle_dir.mkdir(parents=True, exist_ok=True)
shutil.move('kaggle.json', kaggle_dir / 'kaggle.json')
os.chmod(kaggle_dir / 'kaggle.json', 0o600)

os.makedirs(DATA_DIR, exist_ok=True)
!kaggle datasets download -d {KAGGLE_DATASET} -p {DATA_DIR} --unzip

all_images = glob(f'{DATA_DIR}/**/*.jpg', recursive=True) + \
             glob(f'{DATA_DIR}/**/*.jpeg', recursive=True) + \
             glob(f'{DATA_DIR}/**/*.png', recursive=True)

print(f'Imagens encontradas: {len(all_images):,}')

---
## Pseudo-Labels com CLIP + SAM (sem anotacao manual)

Quando nao ha anotacoes manuais disponiveis, este pipeline gera bounding boxes automaticos usando:
- **CLIP** para classificar cada imagem por classe de defeito
- **Grounding DINO** para detectar regioes relevantes com prompts de texto


In [ ]:

!pip install -q groundingdino-py
!pip install -q git+https://github.com/facebookresearch/segment-anything.git

from groundingdino.util.inference import load_model, load_image, predict, annotate
import groundingdino.datasets.transforms as T

print('Grounding DINO instalado.')

In [ ]:

GDINO_CONFIG = '/content/GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py'
GDINO_CKPT   = '/content/groundingdino_swint_ogc.pth'

if not os.path.exists(GDINO_CKPT):
    !wget -q https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth \
        -O {GDINO_CKPT}

print('Pesos baixados.')

In [ ]:
import os
import sys
import shutil
from pathlib import Path
from PIL import Image as PILImage # Renomeado para evitar conflito
from tqdm.notebook import tqdm
from glob import glob

# 1. Uninstall and reinstall groundingdino-py for a clean state.
# This ensures the target file exists in its original (unpatched) form after reinstall.
print("Attempting to uninstall and reinstall groundingdino-py for a clean state...")
!pip uninstall -y groundingdino-py

# Aggressively delete the groundingdino-py installation directory to ensure no old files linger
try:
    # Find the actual groundingdino-py root directory in site-packages/dist-packages
    # This part is a bit heuristic, but generally works in Colab
    for p in sys.path:
        if 'site-packages' in p or 'dist-packages' in p:
            gdino_pkg_path = os.path.join(p, 'groundingdino')
            if os.path.exists(gdino_pkg_path) and os.path.isdir(gdino_pkg_path):
                shutil.rmtree(gdino_pkg_path)
                print(f"Force removed groundingdino package directory: {gdino_pkg_path}")
                break
except Exception as e:
    print(f"Could not force remove groundingdino-py directory: {e}")

!pip install -q groundingdino-py
print("groundingdino-py reinstalled.")

# 2. Apply the patch to the file on disk.
GDINO_BERTWARPER_PATH="/usr/local/lib/python3.12/dist-packages/groundingdino/models/GroundingDINO/bertwarper.py"
print(f"Patching file: {GDINO_BERTWARPER_PATH}")

if os.path.exists(GDINO_BERTWARPER_PATH):
    with open(GDINO_BERTWARPER_PATH, "r") as f:
        source = f.read()

    # The OLD string must include the expected indentation from the file
    OLD = "        self.get_head_mask = bert_model.get_head_mask"
    NEW = (
        "        if hasattr(bert_model, 'get_head_mask'):\n"
        "            self.get_head_mask = bert_model.get_head_mask\n"
        "        else:\n"
        "            self.get_head_mask = lambda *a, **kw: None"
    )

    if OLD in source:
        source = source.replace(OLD, NEW)
        with open(GDINO_BERTWARPER_PATH, "w") as f:
            f.write(source)
        print("✅ Patch applied successfully to bertwarper.py.")
    elif "hasattr(bert_model, 'get_head_mask')" in source:
        print("ℹ️  Patch already applied, skipping.")
    else:
        print("⚠️  Target line not found in bertwarper.py — check the file manually.")
else:
    print(f"❌ File not found: {GDINO_BERTWARPER_PATH}. Patch could not be applied.")

# 3. Clear relevant modules from sys.modules to force a fresh import for GroundingDINO.
print("Clearing groundingdino modules from sys.modules to force fresh import...")
modules_to_delete_gdino = [m for m in sys.modules if m.startswith('groundingdino')]
for m in modules_to_delete_gdino:
    del sys.modules[m]
print("GroundingDINO Modules cleared.")

# 4. Now, re-import everything cleanly and proceed with model loading for GroundingDINO.
from groundingdino.util.inference import load_model, load_image, predict, annotate

# Install transformers for CLIP and load CLIP model
print("Installing transformers for CLIP...")
!pip uninstall -y transformers # Uninstall existing transformers

# Aggressively delete transformers installation directory to ensure no old files linger
try:
    for p in sys.path:
        if 'site-packages' in p or 'dist-packages' in p:
            transformers_pkg_path = os.path.join(p, 'transformers')
            if os.path.exists(transformers_pkg_path) and os.path.isdir(transformers_pkg_path):
                shutil.rmtree(transformers_pkg_path)
                print(f"Force removed transformers package directory: {transformers_pkg_path}")
                break
except Exception as e:
    print(f"Could not force remove transformers directory: {e}")

!pip cache purge # Purge pip cache to ensure fresh download
# Clear transformers modules from sys.modules to force fresh import
print("Clearing transformers modules from sys.modules to force fresh import...")
modules_to_delete_transformers = [m for m in sys.modules if m.startswith('transformers')] or [] # Ensure it's an iterable even if empty
for m in modules_to_delete_transformers:
    del sys.modules[m]
print("Transformers Modules cleared.")
!pip install -q transformers[torch]==4.41.0 # Install a specific stable version with torch extra
from transformers import CLIPProcessor, CLIPModel

print("Loading CLIP model...")
clip_model_name = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(clip_model_name)
clip_processor = CLIPProcessor.from_pretrained(clip_model_name)
print("CLIP model loaded.")


# Clone GroundingDINO if not present (needed for config files, this is separate from the installed package)
if not os.path.exists('/content/GroundingDINO'):
    !git clone https://github.com/IDEA-Research/GroundingDINO.git /content/GroundingDINO

TEXT_PROMPTS = {
    'buraco'                 : 'pothole . hole in road . pavement damage',
    'remendo'                : 'road patch . asphalt repair . pavement patch',
    'trinca'                 : 'road crack . pavement crack . surface crack',
    'sinalizacao_vertical'   : 'road sign . traffic sign . vertical sign',
    'sinalizacao_horizontal' : 'road marking . lane marking . horizontal marking',
    'drenagem'               : 'drainage . gutter . water drain',
    'vegetacao'              : 'vegetation . grass . plants beside road',
}

BOX_THRESHOLD  = 0.35
TEXT_THRESHOLD = 0.25

print("Loading Grounding DINO model...")
gdino_model = load_model(GDINO_CONFIG, GDINO_CKPT)
print("Grounding DINO model loaded.")

def generate_pseudo_label(img_path, class_name, class_idx):
    prompt = TEXT_PROMPTS[class_name]
    image_source, image_tensor = load_image(img_path)
    # Grounding DINO expects a specific image format, ensure image_source is correct after load_image
    h, w = image_source.shape[:2]

    boxes, logits, phrases = predict(
        model=gdino_model,
        image=image_tensor,
        caption=prompt,
        box_threshold=BOX_THRESHOLD,
        text_threshold=TEXT_THRESHOLD,
    )

    yolo_lines = []
    for box in boxes:
        cx, cy, bw, bh = box.tolist()
        yolo_lines.append(f'{class_idx} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')

    return yolo_lines


PSEUDO_IMG_DIR  = Path(YOLO_DATA_DIR) / 'images' / 'train'
PSEUDO_LBL_DIR  = Path(YOLO_DATA_DIR) / 'labels' / 'train'
PSEUDO_IMG_DIR.mkdir(parents=True, exist_ok=True)
PSEUDO_LBL_DIR.mkdir(parents=True, exist_ok=True)

# Get all images from the DATA_DIR, regardless of their subdirectory structure
all_images_paths = glob(f'{DATA_DIR}/**/*.jpg', recursive=True) + \
                   glob(f'{DATA_DIR}/**/*.jpeg', recursive=True) + \
                   glob(f'{DATA_DIR}/**/*.png', recursive=True)

total_generated = 0
# Prepare CLIP class descriptions for classification
clip_class_descriptions = [f"uma imagem de {name}" for name in CLASS_NAMES]

for img_path_str in tqdm(all_images_paths, desc='Classifying and Generating Pseudo-Labels'):
    img_path = Path(img_path_str)

    try:
        # Use CLIP to classify the image first
        image_pil = PILImage.open(img_path).convert("RGB") # Usar PILImage.open()

        inputs = clip_processor(text=clip_class_descriptions, images=image_pil, return_tensors="pt", padding=True)
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()} # Move to GPU if available
        clip_model.to(DEVICE) # Move CLIP model to GPU
        outputs = clip_model(**inputs)
        logits_per_image = outputs.logits_per_image # this is the image-text similarity score
        probs = logits_per_image.softmax(dim=1) # softmax to get probabilities

        best_prob, best_idx = probs.max(dim=1)
        detected_class_name = CLASS_NAMES[best_idx.item()]
        detected_class_idx = best_idx.item()

        # Only proceed if CLIP confidence is high enough for a specific class
        # You can adjust this threshold (e.g., 0.5 or higher) based on your data
        if best_prob.item() > 0.6: # A moderate confidence threshold for CLIP classification
            lines = generate_pseudo_label(str(img_path), detected_class_name, detected_class_idx)
            if not lines:
                # print(f"Grounding DINO found no boxes for {img_path.name} ({detected_class_name})")
                continue

            dest_img = PSEUDO_IMG_DIR / img_path.name
            dest_lbl = PSEUDO_LBL_DIR / (img_path.stem + '.txt')

            shutil.copy(img_path, dest_img)
            dest_lbl.write_text('\n'.join(lines))
            total_generated += 1
        # else:
            # print(f"Skipping {img_path.name} due to low CLIP confidence ({best_prob.item():.2f}) for any class.")

    except Exception as e:
        # print(f"Error processing {img_path.name}: {e}") # Uncomment for debugging specific image errors
        pass

print(f'Pseudo-labels gerados: {total_generated}')
print('Revise e corrija as anotacoes no Roboflow antes do treino final.')

---
## Estrutura do Dataset YOLO e data.yaml

In [ ]:

for split in ['train', 'val', 'test']:
    (Path(YOLO_DATA_DIR) / 'images' / split).mkdir(parents=True, exist_ok=True)
    (Path(YOLO_DATA_DIR) / 'labels' / split).mkdir(parents=True, exist_ok=True)

print(f'Estrutura criada em {YOLO_DATA_DIR}:')
!find {YOLO_DATA_DIR} -type d | sort

In [ ]:
# @title Split das Imagens Anotadas (70/15/15)

from sklearn.model_selection import train_test_split

all_label_files = list((Path(YOLO_DATA_DIR) / 'labels' / 'train').glob('*.txt'))

train_files, temp_files = train_test_split(all_label_files, test_size=0.30,
                                           random_state=SEED)
val_files, test_files   = train_test_split(temp_files, test_size=0.50,
                                           random_state=SEED)

def move_split(label_files, split_name):
    lbl_dst = Path(YOLO_DATA_DIR) / 'labels'  / split_name
    img_dst = Path(YOLO_DATA_DIR) / 'images'  / split_name
    lbl_dst.mkdir(parents=True, exist_ok=True)
    img_dst.mkdir(parents=True, exist_ok=True)

    for lbl in label_files:
        img_src = Path(YOLO_DATA_DIR) / 'images' / 'train' / (lbl.stem + '.jpg')
        if not img_src.exists():
            for ext in ['.jpeg', '.png']:
                img_src = img_src.with_suffix(ext)
                if img_src.exists(): break
        if img_src.exists():
            shutil.move(str(img_src), img_dst / img_src.name)
        if split_name != 'train':
            shutil.move(str(lbl), lbl_dst / lbl.name)

move_split(val_files,  'val')
move_split(test_files, 'test')

print('Split realizado.')
print(f'  Treino : {len(list((Path(YOLO_DATA_DIR)/"labels"/"train").glob("*.txt"))):,}')
print(f'  Val    : {len(val_files):,}')
print(f'  Teste  : {len(test_files):,}')

In [ ]:
import os
from pathlib import Path

train_image_dir = Path(YOLO_DATA_DIR) / 'images' / 'train'
image_files = list(train_image_dir.glob('*'))

print(f"Found {len(image_files)} images in {train_image_dir}:")
for i, img_file in enumerate(image_files[:5]):
    print(f"- {img_file.name}")
if len(image_files) > 5:
    print("... (and {len(image_files) - 5} more)")


In [ ]:
# @title Gera data.yaml

YAML_PATH = os.path.join(YOLO_DATA_DIR, 'data.yaml')

yaml_content = {
    'path'  : YOLO_DATA_DIR,
    'train' : 'images/train',
    'val'   : 'images/val',
    'test'  : 'images/test',
    'nc'    : NUM_CLASSES,
    'names' : CLASS_NAMES,
}

with open(YAML_PATH, 'w') as f:
    yaml.dump(yaml_content, f, default_flow_style=False, allow_unicode=True)

print(f'data.yaml gerado em: {YAML_PATH}')
with open(YAML_PATH) as f:
    print(f.read())

In [ ]:
# @title Visualizacao das Anotacoes (verificacao)

COLORS = plt.cm.get_cmap('tab10', NUM_CLASSES).colors

def draw_yolo_boxes(img_path, lbl_path):
    img = np.array(PILImage.open(img_path).convert('RGB')) # Usar PILImage.open()
    h, w = img.shape[:2]
    fig, ax = plt.subplots(1, 1, figsize=(8, 5))
    ax.imshow(img)

    if Path(lbl_path).exists():
        with open(lbl_path) as f:
            lines = f.readlines()
        for line in lines:
            parts = line.strip().split()
            if len(parts) != 5: continue
            cls_idx = int(parts[0])
            cx, cy, bw, bh = map(float, parts[1:])
            x1 = (cx - bw / 2) * w
            y1 = (cy - bh / 2) * h
            rect = patches.Rectangle(
                (x1, y1), bw * w, bh * h,
                linewidth=2, edgecolor=COLORS[cls_idx % len(COLORS)], facecolor='none'
            )
            ax.add_patch(rect)
            ax.text(x1, y1 - 4, CLASS_NAMES[cls_idx],
                    color='white', fontsize=9, fontweight='bold',
                    bbox=dict(facecolor=COLORS[cls_idx % len(COLORS)], alpha=0.8, pad=2))

    ax.axis('off')
    ax.set_title(Path(img_path).name)
    return fig

train_imgs = list((Path(YOLO_DATA_DIR) / 'images' / 'train').glob('*.*'))[:4]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, img_path in zip(axes.flatten(), train_imgs):
    lbl_path = Path(YOLO_DATA_DIR) / 'labels' / 'train' / (img_path.stem + '.txt')
    img   = np.array(PILImage.open(img_path).convert('RGB')) # Usar PILImage.open()
    h, w  = img.shape[:2]
    ax.imshow(img)

    if lbl_path.exists():
        for line in lbl_path.read_text().splitlines():
            parts = line.strip().split()
            if len(parts) != 5: continue
            cls_idx = int(parts[0])
            cx, cy, bw, bh = map(float, parts[1:])
            x1 = (cx - bw / 2) * w
            y1 = (cy - bh / 2) * h
            rect = patches.Rectangle(
                (x1, y1), bw * w, bh * h,
                linewidth=2, edgecolor=COLORS[cls_idx % len(COLORS)], facecolor='none'
            )
            ax.add_patch(rect)
            ax.text(x1, y1 - 4, CLASS_NAMES[cls_idx],
                    color='white', fontsize=8, fontweight='bold',
                    bbox=dict(facecolor=COLORS[cls_idx % len(COLORS)], alpha=0.8, pad=1))
    ax.axis('off')
    ax.set_title(img_path.name, fontsize=9)

plt.suptitle('Verificacao das Anotacoes YOLO — Treino', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/anotacoes_preview.png', bbox_inches='tight', dpi=120)
plt.show()


---
## Treinamento YOLOv8

In [ ]:

model = YOLO(YOLO_MODEL)

print(f'Modelo carregado: {YOLO_MODEL}')
print(model.info())

In [ ]:
# Treino

RUN_DIR = os.path.join(MODEL_DIR, 'runs')

# Re-initialize model to ensure overrides are correctly populated
model = YOLO(YOLO_MODEL)

results = model.train(
    data        = YAML_PATH,
    epochs      = NUM_EPOCHS,
    imgsz       = IMG_SIZE,
    batch       = BATCH_SIZE,
    device      = DEVICE,
    project     = RUN_DIR,
    name        = 'pavimentos_yolov8',
    seed        = SEED,
    patience    = 15,
    lr0         = 0.01,
    lrf         = 0.01,
    momentum    = 0.937,
    weight_decay= 0.0005,
    warmup_epochs     = 3,
    warmup_momentum   = 0.8,
    box               = 7.5,
    cls               = 0.5,
    dfl               = 1.5,
    flipud      = 0.0,
    fliplr      = 0.5,
    mosaic      = 1.0,
    mixup       = 0.1,
    copy_paste  = 0.1,
    hsv_h       = 0.015,
    hsv_s       = 0.7,
    hsv_v       = 0.4,
    degrees     = 10.0,
    translate   = 0.1,
    scale       = 0.5,
    shear       = 2.0,
    perspective = 0.0,
    save        = True,
    save_period = 10,
    plots       = True,
    verbose     = True,
)

BEST_WEIGHTS = results.save_dir / 'weights' / 'best.pt'
print(f'Treinamento concluido. Melhor modelo: {BEST_WEIGHTS}')

In [ ]:
results_csv = Path(results.save_dir) / 'results.csv'
df_res = pd.read_csv(results_csv)
df_res.columns = df_res.columns.str.strip()

print("Columns in results.csv:", df_res.columns.tolist())

metrics_pairs = [
    ('train/box_loss', 'val/box_loss',   'Box Loss'),
    ('train/cls_loss', 'val/cls_loss',   'Class Loss'),
    ('metrics/mAP50(B)',  'metrics/mAP50-95(B)','mAP'), # Updated column names
    ('metrics/precision(B)', 'metrics/recall(B)', 'Precision / Recall'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, (col1, col2, title) in zip(axes.flatten(), metrics_pairs):
    current_handles = []
    current_labels = []

    if col1 in df_res.columns:
        line1, = ax.plot(df_res['epoch'], df_res[col1],
                        color='steelblue', linewidth=2)
        current_handles.append(line1)
        current_labels.append(col1.split('/')[-1])
    if col2 in df_res.columns:
        line2, = ax.plot(df_res['epoch'], df_res[col2],
                        color='coral', linewidth=2, linestyle='--')
        current_handles.append(line2)
        current_labels.append(col2.split('/')[-1])

    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Epoca')
    if current_handles: # Only add legend if there are lines to show
        ax.legend(handles=current_handles, labels=current_labels) # Explicitly pass handles and labels
    ax.grid(alpha=0.3)

plt.suptitle('Curvas de Treinamento — YOLOv8', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/yolo_training_curves.png', bbox_inches='tight', dpi=150)
plt.show()

---
## Avaliacao no Conjunto de Teste

In [ ]:
# Metricas  de Teste

best_model = YOLO(BEST_WEIGHTS)

test_metrics = best_model.val(
    data    = YAML_PATH,
    split   = 'test',
    imgsz   = IMG_SIZE,
    conf    = CONF_THRESH,
    iou     = IOU_THRESH,
    device  = DEVICE,
    verbose = True,
)

print('\nResultados no conjunto de teste:')
print(f'  mAP@0.50      : {test_metrics.box.map50:.4f}')
print(f'  mAP@0.50:0.95  : {test_metrics.box.map:.4f}')
print(f'  Precision      : {test_metrics.box.mp:.4f}')
print(f'  Recall         : {test_metrics.box.mr:.4f}')

### Analise da Matriz de Confusao

A matriz de confusão é uma ferramenta essencial para entender o desempenho do modelo em cada classe. Ela mostra onde o modelo está acertando (verdadeiros positivos) e onde está errando (falsos positivos e falsos negativos).

Abaixo, é carregada e exibida a matriz de confusão que foi gerada automaticamente pelo YOLOv8 durante a etapa de validação (`best_model.val`).

In [ ]:
from IPython.display import Image, display

# O caminho para o diretório de salvamento dos resultados do treino
save_dir = Path(results.save_dir)

# O YOLOv8 salva a matriz de confusão como 'confusion_matrix.png' dentro do diretório de validação
confusion_matrix_path = save_dir / 'confusion_matrix.png'

if confusion_matrix_path.exists():
    print(f"Exibindo a matriz de confusão de: {confusion_matrix_path}")
    display(Image(filename=confusion_matrix_path))
else:
    print(f"Arquivo da matriz de confusão não encontrado em: {confusion_matrix_path}")
    print("Por favor, verifique se 'plots=True' foi definido no comando `model.train()` ou `best_model.val()`.")

#### Como interpretar a matriz de confusão:

*   **Eixo Y (Real):** Representa as classes verdadeiras (ground truth).
*   **Eixo X (Previsto):** Representa as classes que o modelo previu.
*   **Diagonal Principal:** Os valores na diagonal principal (de cima para baixo, da esquerda para a direita) indicam o número de predições corretas para cada classe (verdadeiros positivos).
*   **Valores Fora da Diagonal Principal:**
    *   **Linha:** Se você olhar para uma linha específica (por exemplo, classe 'buraco'), os valores fora da diagonal representam os **falsos negativos** (instâncias de 'buraco' que o modelo classificou incorretamente como outras classes).
    *   **Coluna:** Se você olhar para uma coluna específica (por exemplo, classe 'trinca'), os valores fora da diagonal representam os **falsos positivos** (instâncias de outras classes que o modelo classificou incorretamente como 'trinca').

Ao analisar essa matriz, você pode identificar quais classes o modelo está confundindo mais frequentemente e direcionar seus esforços para melhorar essas classificações.

In [ ]:
# Fix: Ensure 'test_metrics' is defined.
# If this cell is run out of order or after a kernel restart,
# 'test_metrics' might not be in the current scope.
# We re-evaluate it here to ensure the plot can be generated.
if 'test_metrics' not in globals():
    print("Warning: 'test_metrics' not found in current scope. Re-running model validation.")
    from ultralytics import YOLO # Ensure YOLO is imported
    best_model = YOLO(BEST_WEIGHTS)
    test_metrics = best_model.val(
        data    = YAML_PATH,
        split   = 'test',
        imgsz   = IMG_SIZE,
        conf    = CONF_THRESH,
        iou     = IOU_THRESH,
        device  = DEVICE,
        verbose = False, # Set to False to reduce output if re-running
    )

# mAP por Classe
per_class_ap50 = test_metrics.box.ap50

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(CLASS_NAMES[:len(per_class_ap50)], per_class_ap50,
               color=sns.color_palette('husl', len(per_class_ap50)))
ax.axvline(x=test_metrics.box.map50, color='red', linestyle='--',
           linewidth=1.5, label=f'mAP@50 medio = {test_metrics.box.map50:.3f}')
ax.set_xlabel('AP@0.50')
ax.set_title('Average Precision por Classe (AP@0.50)', fontsize=13, fontweight='bold')
ax.legend()
ax.set_xlim(0, 1)
for bar, val in zip(bars, per_class_ap50):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('/content/yolo_ap_por_classe.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Inferencia Visual em Imagens do Conjunto de Teste

test_imgs = list((Path(YOLO_DATA_DIR) / 'images' / 'test').glob('*.*'))[:6]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, img_path in zip(axes.flatten(), test_imgs):
    result = best_model.predict(
        source   = str(img_path),
        conf     = CONF_THRESH,
        iou      = IOU_THRESH,
        device   = DEVICE,
        verbose  = False,
    )[0]

    annotated = result.plot()
    annotated = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    ax.imshow(annotated)
    n_det = len(result.boxes)
    ax.set_title(f'{img_path.name}\n{n_det} deteccao(oes)', fontsize=9)
    ax.axis('off')

plt.suptitle('Inferencia YOLOv8 — Conjunto de Teste', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/yolo_inferencia_teste.png', bbox_inches='tight', dpi=150)
plt.show()

### Gráfico Comparativo das Perdas de Treino e Validação

Este gráfico foca especificamente na evolução do `Box Loss` (erro na localização da caixa delimitadora) e do `Class Loss` (erro na classificação da classe) tanto para o conjunto de treino quanto para o de validação ao longo das épocas. Isso ajuda a identificar rapidamente se o modelo está aprendendo corretamente e se há sinais de *overfitting* ou *underfitting*.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pathlib import Path

# Certifica que df_res está disponível, recarregando se necessário
if 'df_res' not in globals():
    results_csv = Path(results.save_dir) / 'results.csv'
    df_res = pd.read_csv(results_csv)
    df_res.columns = df_res.columns.str.strip()

loss_metrics = [
    ('train/box_loss', 'val/box_loss',   'Box Loss'),
    ('train/cls_loss', 'val/cls_loss',   'Class Loss'),
]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Curvas de Perda de Treino e Validação — YOLOv8', fontsize=16, fontweight='bold')

for ax, (train_col, val_col, title) in zip(axes, loss_metrics):
    if train_col in df_res.columns and val_col in df_res.columns:
        ax.plot(df_res['epoch'], df_res[train_col], label=f'Treino {title}',
                color='steelblue', linewidth=2)
        ax.plot(df_res['epoch'], df_res[val_col], label=f'Validação {title}',
                color='coral', linewidth=2, linestyle='--')
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('Época')
        ax.set_ylabel('Perda')
        ax.legend()
        ax.grid(alpha=0.3)
    else:
        ax.text(0.5, 0.5, f'Dados para {title} não encontrados.',
                horizontalalignment='center', verticalalignment='center',
                transform=ax.transAxes, fontsize=12, color='gray')
        ax.set_title(title, fontweight='bold')
        ax.axis('off')

plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Ajusta layout para evitar sobreposição do título principal
plt.savefig('/content/yolo_loss_curves.png', bbox_inches='tight', dpi=150)
plt.show()

---
## Calculo do ICM a partir das Deteccoes

In [ ]:
ICM_PESO = {
    'buraco'                 : 5,
    'remendo'                : 2,
    'trinca'                 : 3,
    'sinalizacao_vertical'   : 1,
    'sinalizacao_horizontal' : 1,
    'drenagem'               : 2,
    'vegetacao'              : 1,
}


def calcular_icm(img_path, model, conf=CONF_THRESH, iou=IOU_THRESH):
    result = model.predict(
        source  = str(img_path),
        conf    = conf,
        iou     = iou,
        device  = DEVICE,
        verbose = False,
    )[0]

    img      = PILImage.open(img_path) # Usar PILImage.open()
    img_area = img.width * img.height

    defeitos = {}
    for box in result.boxes:
        cls_idx  = int(box.cls.item())
        cls_name = CLASS_NAMES[cls_idx]
        xyxy     = box.xyxy[0].cpu().numpy()
        area_px  = (xyxy[2] - xyxy[0]) * (xyxy[3] - xyxy[1])
        area_pct = area_px / img_area * 100

        if cls_name not in defeitos:
            defeitos[cls_name] = {'count': 0, 'area_total_pct': 0.0}
        defeitos[cls_name]['count']          += 1
        defeitos[cls_name]['area_total_pct'] += area_pct

    penalidade = 0.0
    for cls_name, info in defeitos.items():
        penalidade += ICM_PESO.get(cls_name, 1) * info['area_total_pct']

    icm_score = max(0, 100 - penalidade)

    if icm_score >= 80:
        condicao = 'Otimo'
    elif icm_score >= 60:
        condicao = 'Bom'
    elif icm_score >= 40:
        condicao = 'Regular'
    elif icm_score >= 20:
        condicao = 'Ruim'
    else:
        condicao = 'Pessimo'

    return {
        'imagem'    : Path(img_path).name,
        'icm_score' : round(icm_score, 2),
        'condicao'  : condicao,
        'defeitos'  : defeitos,
        'n_deteccoes': len(result.boxes),
    }


print('Funcao de calculo do ICM definida.')


In [ ]:
icm_results = []

for img_path in tqdm(test_imgs, desc='Calculando ICM'):
    try:
        r = calcular_icm(img_path, best_model)
        icm_results.append(r)
    except Exception as e:
        # Modified to print the actual error instead of silently passing
        print(f"Erro ao calcular ICM para {img_path.name}: {e})")

icm_df = pd.DataFrame([{ # type: ignore
    'imagem'      : r['imagem'],
    'icm_score'   : r['icm_score'],
    'condicao'    : r['condicao'],
    'n_deteccoes' : r['n_deteccoes'],
} for r in icm_results])

# Added a check to see if the DataFrame is empty before trying to access columns
if not icm_df.empty:
    print(icm_df.to_string(index=False))
    print(f'\nICM medio do trecho: {icm_df["icm_score"].mean():.2f}')
else:
    print("\nNenhum resultado de ICM foi gerado. Verifique os erros acima.")

In [ ]:
condicao_cores = {
    'Otimo'   : '#2ecc71',
    'Bom'     : '#27ae60',
    'Regular' : '#f39c12',
    'Ruim'    : '#e67e22',
    'Pessimo' : '#e74c3c',
}

# Verifique se icm_df não está vazio antes de tentar acessá-lo
if not icm_df.empty:
    fig, ax = plt.subplots(figsize=(12, 5))
    cores = [condicao_cores.get(c, 'gray') for c in icm_df['condicao']]
    bars  = ax.bar(icm_df['imagem'], icm_df['icm_score'], color=cores, edgecolor='white')

    ax.axhline(y=80, color='green',  linestyle='--', linewidth=1, alpha=0.7, label='Otimo (80)')
    ax.axhline(y=60, color='orange', linestyle='--', linewidth=1, alpha=0.7, label='Regular (60)')
    ax.axhline(y=40, color='red',    linestyle='--', linewidth=1, alpha=0.7, label='Ruim (40)')

    ax.set_xlabel('Imagem')
    ax.set_ylabel('ICM Score')
    ax.set_title('Indice de Condicao de Manutencao (ICM) por Imagem', fontsize=13, fontweight='bold')
    ax.set_ylim(0, 110)
    ax.tick_params(axis='x', rotation=45)
    ax.legend(loc='lower left')

    for bar, row in zip(bars, icm_df.itertuples()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{row.icm_score:.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

    plt.tight_layout()
    plt.savefig('/content/icm_scores.png', bbox_inches='tight', dpi=150)
    plt.show()
else:
    print("Não há dados no DataFrame icm_df para gerar o gráfico.")


---
## Exportacao do Modelo

In [ ]:

ONNX_PATH = os.path.join(MODEL_DIR, 'pavimentos_yolov8.onnx')

best_model.export(
    format   = 'onnx',
    imgsz    = IMG_SIZE,
    opset    = 12,
    simplify = True,
    dynamic  = False,
)

exported_onnx = str(Path(BEST_WEIGHTS).with_suffix('.onnx'))
if os.path.exists(exported_onnx):
    shutil.copy(exported_onnx, ONNX_PATH)

print(f'ONNX exportado: {ONNX_PATH}')

In [ ]:
# Salva Metadados e CSV de Resultados

CSV_ICM   = os.path.join(MODEL_DIR, 'resultados_icm.csv')
META_PATH = os.path.join(MODEL_DIR, 'model_metadata.json')

icm_df.to_csv(CSV_ICM, index=False)

metadata = {
    'modelo'        : 'YOLOv8n',
    'arquitetura'   : 'YOLOv8',
    'dataset'       : 'Pavimentos-Brasil (Kaggle)',
    'classes'       : CLASS_NAMES,
    'num_classes'   : NUM_CLASSES,
    'img_size'      : IMG_SIZE,
    'conf_thresh'   : CONF_THRESH,
    'iou_thresh'    : IOU_THRESH,
    'mAP50'         : float(test_metrics.box.map50),
    'mAP50_95'      : float(test_metrics.box.map),
    'precision'     : float(test_metrics.box.mp),
    'recall'        : float(test_metrics.box.mr),
    'pesos_icm'     : ICM_PESO,
}

with open(META_PATH, 'w') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f'CSV salvo    : {CSV_ICM}')
print(f'Metadados    : {META_PATH}')
print(f'Modelo ONNX  : {ONNX_PATH}')
print(f'Pesos .pt    : {BEST_WEIGHTS}')

---
## Resumo Final

| Etapa | Descricao |
|-------|-----------|
| Modelo | YOLOv8n fine-tuned no Pavimentos-Brasil |
| Saida | Bounding boxes + classe + confianca por defeito |
| ICM | Calculado pela area relativa dos defeitos x peso por classe |
| Exportacao | ONNX (producao), .pt (PyTorch), CSV com scores |

**Proximos passos naturais:**
- Usar `YOLOv8s` ou `YOLOv8m` para ganhar acuracia (mais parametros)
- Integrar inferencia em video (inspecao por camera veicular)
- Deploy como API REST com FastAPI + Docker